In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import time
import gc
import pickle
from datetime import datetime
import cmaps
import os,sys
from pathlib import Path
WAVE_TOOLS_PATH = Path("/work/mh1498/m301257/wave_tools")
sys.path.insert(0, str(WAVE_TOOLS_PATH.parent))
from wave_tools import CCKWFilter
import matplotlib.ticker as ticker
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
# 配置参数
CACHE_DIR = "./cache/kelvin_wave/"
REGRESSION_CACHE_DIR = os.path.join(CACHE_DIR, "latent_heat_flux_with_lon")
FIG_SAVE_DIR = "./figures/latent_heat_flux_composite/"

# 创建必要的目录
for d in [CACHE_DIR, REGRESSION_CACHE_DIR, FIG_SAVE_DIR]:
    os.makedirs(d, exist_ok=True)

# 分析参数
LEAD_LAG_DAYS = 4  
EQUATOR_LAT = slice(5, 10)     # 用于纬向平均的纬度范围（5-10°N）

# 实验列表
EXPERIMENTS = ['CNTL', 'P4K', '4CO2']

def load_filtered_data(var_name, exp_list=None):
    """
    加载滤波后的数据
    
    Parameters:
    -----------
    var_name : str
        变量名称 ('pr', 'ua', 'va', 'omega')
    exp_list : list, optional
        实验列表
    
    Returns:
    --------
    data_dict : dict
        {exp_name: xr.DataArray}
    """
    if exp_list is None:
        exp_list = EXPERIMENTS
    
    data_dict = {}
    
    # 定义可能的缓存目录（按优先级顺序）
    possible_dirs = [
        CACHE_DIR,                                    # ./cache/kelvin_wave/
        "./cache/kelvin_wave_3d/",                    # 3D变量目录
        os.path.join(CACHE_DIR, "../kelvin_wave_3d/") # 相对路径
    ]
    
    print(f"\n📂 Loading {var_name} data...")
    for exp in exp_list:
        loaded = False
        
        # 尝试在不同目录中查找文件
        for cache_dir in possible_dirs:
            cache_file = os.path.join(cache_dir, f'kelvin_{var_name}_{exp.lower()}.nc')
            
            if os.path.exists(cache_file):
                try:
                    data = xr.open_dataarray(cache_file)
                    
                    # 标准化维度名称：将 'level' 重命名为 'lev'（如果存在）
                    if 'level' in data.dims:
                        data = data.rename({'level': 'lev'})
                    
                    data_dict[exp] = data
                    file_size = os.path.getsize(cache_file) / (1024**2)
                    print(f"  ✅ {exp}: Loaded from {os.path.basename(cache_dir)}, "
                          f"Shape={data.shape}, Size={file_size:.1f}MB")
                    loaded = True
                    break
                except Exception as e:
                    print(f"  ❌ {exp}: Error loading from {cache_dir} - {str(e)}")
        
        if not loaded:
            print(f"  ⚠️  {exp}: Cache file not found in any directory!")
    
    return data_dict

print("✅ load_filtered_data 函数定义完成")

✅ load_filtered_data 函数定义完成


In [4]:


def _ocean(ds):
    fraction = xr.open_dataarray(r'../processed_data/land_mask_2deg.nc')
    return fraction == 0
ocean_mask = _ocean(None)
# 步骤2: 加载潜热通量数据
lhf_data_cntl = xr.open_dataarray('/work/mh1498/m301257/processed_data/2d_layers/hfls_cntl/hfls_2deg_interp.nc').where(_ocean, drop=False)
lhf_data_p4k = xr.open_dataarray('/work/mh1498/m301257/processed_data/2d_layers/hfls_p4k/hfls_2deg_interp.nc')  .where(_ocean, drop=False)
lhf_data_4co2 = xr.open_dataarray('/work/mh1498/m301257/processed_data/2d_layers/hfls_4co2/hfls_2deg_interp.nc').where(_ocean, drop=False)

lhf_data_cntl = xr.where(np.isinf(lhf_data_cntl), np.nan, lhf_data_cntl)
lhf_data_p4k = xr.where(np.isinf(lhf_data_p4k), np.nan, lhf_data_p4k)
lhf_data_4co2 = xr.where(np.isinf(lhf_data_4co2), np.nan, lhf_data_4co2)

# 步骤3: kf-filter
def kf_filter_lhf(lhf_data):
    """
    对潜热通量数据应用CCKW滤波器
    """
    
    print("\n🎛️  Applying CCKW filter to latent heat flux data...")
    wave_filter = CCKWFilter(
    ds=lhf_data.fillna(0),
    # sel_dict={'time': slice('1980-01-01', '1993-12-31'), 'lat': slice(-15, 15)},
    wave_name='kelvin',
    units='w/m^2',
    spd=1,
    n_workers=4
)

    

    # 方式2：一键执行（推荐）
    filtered_data = wave_filter.process()
    
    return filtered_data    

lhf_kelvin_cntl = kf_filter_lhf(lhf_data_cntl*(-1))
lhf_kelvin_p4k  = kf_filter_lhf(lhf_data_p4k*(-1))
lhf_kelvin_4co2 = kf_filter_lhf(lhf_data_4co2*(-1))



🎛️  Applying CCKW filter to latent heat flux data...
🌊 Processing KELVIN wave filter

==================== Loaded Data Information ====================
Type: <class 'xarray.core.dataarray.DataArray'>
Shape: (5114, 15, 180)
Data type: float64
First few values: <xarray.DataArray 'hfls' (time: 5, lat: 15, lon: 180)> Size: 108kB
dask.array<getitem, shape=(5, 15, 180), dtype=float64, chunksize=(5, 15, 180), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 40B 1980-01-01 1980-01-02 ... 1980-01-05
  * lat      (lat) float64 120B -14.0 -12.0 -10.0 -8.0 ... 8.0 10.0 12.0 14.0
  * lon      (lon) float64 1kB 0.0 2.0 4.0 6.0 8.0 ... 352.0 354.0 356.0 358.0
⏳ Detrending data...
⏳ Performing FFT...
⏳ Applying filter...
⏳ Performing inverse FFT...
⏳ Creating output...
✅ KELVIN wave filtering completed!


🎛️  Applying CCKW filter to latent heat flux data...
🌊 Processing KELVIN wave filter

==================== Loaded Data Information ====================
Type: <class 'xarray.c

In [5]:
# 1. 📂 加载事件
lon_events_dict, lon_lags, success = load_longitude_events()
lon_events_dict


📂 加载经度事件数据
文件: ../composite_data/longitude_events.json

✅ 经度事件数据加载成功！

📊 事件数量统计:
   CNTL: 11071 个事件
   P4K: 11023 个事件
   4CO2: 11118 个事件

📏 经度偏移配置:
   范围: -30° 到 30°
   间隔: 10°
   总点数: 7


{'CNTL': [{'time_idx': 0,
   'lon_idx': 40,
   'lon': 260.0,
   'intensity': 0.6746160761120732},
  {'time_idx': 0,
   'lon_idx': 54,
   'lon': 288.0,
   'intensity': 0.39211053121659883},
  {'time_idx': 1, 'lon_idx': 44, 'lon': 268.0, 'intensity': 0.639982787130486},
  {'time_idx': 2,
   'lon_idx': 47,
   'lon': 274.0,
   'intensity': 0.5940773197015164},
  {'time_idx': 3,
   'lon_idx': 51,
   'lon': 282.0,
   'intensity': 0.5725546979599847},
  {'time_idx': 4,
   'lon_idx': 55,
   'lon': 290.0,
   'intensity': 0.5536404253497622},
  {'time_idx': 5,
   'lon_idx': 2,
   'lon': 184.0,
   'intensity': 0.44303200441991575},
  {'time_idx': 5,
   'lon_idx': 18,
   'lon': 216.0,
   'intensity': 0.06912563852969458},
  {'time_idx': 6, 'lon_idx': 7, 'lon': 194.0, 'intensity': 0.5162918883900933},
  {'time_idx': 7,
   'lon_idx': 11,
   'lon': 202.0,
   'intensity': 0.41325439722812446},
  {'time_idx': 7,
   'lon_idx': 27,
   'lon': 234.0,
   'intensity': 0.06035156430284601},
  {'time_idx': 8, 

In [6]:


#  加载比湿数据
diff_qa_qs = {
    'CNTL': xr.open_dataset(os.path.join('/work/mh1498/m301257/3D_data', 'difference_qs_qa.nc'))['diff_qs-qa_cntl'].where(ocean_mask, drop=False), 
    '4CO2': xr.open_dataset(os.path.join('/work/mh1498/m301257/3D_data', 'difference_qs_qa.nc'))['diff_qs-qa_4co2'].where(ocean_mask, drop=False),
    'P4K': xr.open_dataset(os.path.join('/work/mh1498/m301257/3D_data', 'difference_qs_qa.nc'))['diff_qs-qa_p4k'].where(ocean_mask, drop=False)
}


surface_wind = {

    "CNTL" : xr.open_dataset(f"/work/mh1498/m301257/processed_data/2d_layers/sfcwind_cntl/sfcwind_2deg_interp.nc")['sfcwind'].where(ocean_mask, drop=False),
    "4CO2" : xr.open_dataset(f"/work/mh1498/m301257/processed_data/2d_layers/sfcwind_4co2/sfcwind_2deg_interp.nc")['sfcwind'].where(ocean_mask, drop=False),
    "P4K" : xr.open_dataset(f"/work/mh1498/m301257/processed_data/2d_layers/sfcwind_p4k/sfcwind_2deg_interp.nc")['sfcwind'].where(ocean_mask, drop=False)
}
# 步骤3: kf-filter
def kf_filter_diff_qa_qs(data):
    
    print("\n🎛️  Applying CCKW filter to latent heat flux data...")
    wave_filter = CCKWFilter(
    ds=data.fillna(0),
    # sel_dict={'time': slice('1980-01-01', '1993-12-31'), 'lat': slice(-15, 15)},
    wave_name='kelvin',
    units='w/m^2',
    spd=1,
    n_workers=4)

    

    # 方式2：一键执行（推荐）
    filtered_data = wave_filter.process()
    
    return filtered_data    

diff_qs_qa_kelvin_cntl = kf_filter_diff_qa_qs(diff_qa_qs['CNTL'])
diff_qs_qa_kelvin_p4k  = kf_filter_diff_qa_qs(diff_qa_qs['P4K'])
diff_qs_qa_kelvin_4co2 = kf_filter_diff_qa_qs(diff_qa_qs['4CO2'])
surface_wind_kelvin_cntl = kf_filter_diff_qa_qs(surface_wind['CNTL'])
surface_wind_kelvin_p4k  = kf_filter_diff_qa_qs(surface_wind['P4K'])
surface_wind_kelvin_4co2 = kf_filter_diff_qa_qs(surface_wind['4CO2'])


🎛️  Applying CCKW filter to latent heat flux data...
🌊 Processing KELVIN wave filter

==================== Loaded Data Information ====================
Type: <class 'xarray.core.dataarray.DataArray'>
Shape: (5114, 15, 180)
Data type: float64
First few values: <xarray.DataArray 'diff_qs-qa_cntl' (time: 5, lat: 15, lon: 180)> Size: 108kB
dask.array<getitem, shape=(5, 15, 180), dtype=float64, chunksize=(5, 15, 180), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 40B 1980-01-01 1980-01-02 ... 1980-01-05
  * lat      (lat) float64 120B -14.0 -12.0 -10.0 -8.0 ... 8.0 10.0 12.0 14.0
  * lon      (lon) float64 1kB 0.0 2.0 4.0 6.0 8.0 ... 352.0 354.0 356.0 358.0
⏳ Detrending data...
⏳ Performing FFT...
⏳ Applying filter...
⏳ Performing inverse FFT...
⏳ Creating output...
✅ KELVIN wave filtering completed!


🎛️  Applying CCKW filter to latent heat flux data...
🌊 Processing KELVIN wave filter

==================== Loaded Data Information ====================
Type: <clas

In [7]:

# 步骤3: 使用相同的事件进行合成
lhf_composite_cntl, lhf_std_cntl, n_events_cntl = compute_longitude_composite_vectorized(
    data=lhf_kelvin_cntl,
    events=lon_events_dict['CNTL'],  # 使用相同的事件！
    lon_lags=lon_lags                 # 使用相同的偏移！
)

lhf_composite_p4k, lhf_std_p4k, n_events_p4k = compute_longitude_composite_vectorized(
    data=lhf_kelvin_p4k,
    events=lon_events_dict['P4K'],  # 使用相同的事件！
    lon_lags=lon_lags                 # 使用相同的偏移！
)

lhf_composite_4co2, lhf_std_4co2, n_events_4co2 = compute_longitude_composite_vectorized(
    data=lhf_kelvin_4co2,
    events=lon_events_dict['4CO2'],  # 使用相同的事件！
    lon_lags=lon_lags                 # 使用相同的偏移！
)

diff_qs_qa_kelvin_cntl,_,_ = compute_longitude_composite_vectorized(
    data = diff_qs_qa_kelvin_cntl,
    events=lon_events_dict['CNTL'],  # 使用相同的事件！
    lon_lags=lon_lags                 # 使用相同的偏移！  
)

diff_qs_qa_kelvin_p4k,_,_ = compute_longitude_composite_vectorized(
    data = diff_qs_qa_kelvin_p4k,
    events=lon_events_dict['P4K'],  # 使用相同的事件！
    lon_lags=lon_lags                 # 使用相同的偏移！  
)

diff_qs_qa_kelvin_4co2,_,_ = compute_longitude_composite_vectorized(
    data = diff_qs_qa_kelvin_4co2,
    events=lon_events_dict['4CO2'],  # 使用相同的事件！
    lon_lags=lon_lags                 # 使用相同的偏移！  
)

surface_wind_kelvin_cntl,_,_ = compute_longitude_composite_vectorized(
    data = surface_wind_kelvin_cntl,
    events=lon_events_dict['CNTL'],  # 使用相同的事件！
    lon_lags=lon_lags                 # 使用相同的偏移！  
)   

surface_wind_kelvin_p4k,_,_ = compute_longitude_composite_vectorized(
    data = surface_wind_kelvin_p4k,
    events=lon_events_dict['P4K'],  # 使用相同的事件！
    lon_lags=lon_lags                 # 使用相同的偏移！  
)   

surface_wind_kelvin_4co2,_,_ = compute_longitude_composite_vectorized(
    data = surface_wind_kelvin_4co2,
    events=lon_events_dict['4CO2'],  # 使用相同的事件！     
    lon_lags=lon_lags                 # 使用相同的偏移！  
)   


⚡ 计算经度合成（矢量化版本 - 高性能）
   事件数: 11071
   经度偏移: [-30, -20, -10, 0, 10, 20, 30]

✅ 合成完成！
   有效事件数: 11071
   合成形状: (7, 15)
   数值范围: [-1.2759, 1.0628]
   ⚡ 耗时: 0.27 秒

⚡ 计算经度合成（矢量化版本 - 高性能）
   事件数: 11023
   经度偏移: [-30, -20, -10, 0, 10, 20, 30]

✅ 合成完成！
   有效事件数: 11023
   合成形状: (7, 15)
   数值范围: [-1.2192, 1.1694]
   ⚡ 耗时: 0.27 秒

⚡ 计算经度合成（矢量化版本 - 高性能）
   事件数: 11118
   经度偏移: [-30, -20, -10, 0, 10, 20, 30]

✅ 合成完成！
   有效事件数: 11118
   合成形状: (7, 15)
   数值范围: [-1.1236, 0.9615]
   ⚡ 耗时: 0.25 秒

⚡ 计算经度合成（矢量化版本 - 高性能）
   事件数: 11071
   经度偏移: [-30, -20, -10, 0, 10, 20, 30]

✅ 合成完成！
   有效事件数: 11071
   合成形状: (7, 15)
   数值范围: [-0.0000, 0.0000]
   ⚡ 耗时: 0.26 秒

⚡ 计算经度合成（矢量化版本 - 高性能）
   事件数: 11023
   经度偏移: [-30, -20, -10, 0, 10, 20, 30]

✅ 合成完成！
   有效事件数: 11023
   合成形状: (7, 15)
   数值范围: [-0.0001, 0.0000]
   ⚡ 耗时: 0.26 秒

⚡ 计算经度合成（矢量化版本 - 高性能）
   事件数: 11118
   经度偏移: [-30, -20, -10, 0, 10, 20, 30]

✅ 合成完成！
   有效事件数: 11118
   合成形状: (7, 15)
   数值范围: [-0.0000, 0.0000]
   ⚡ 耗时: 0.25 秒

⚡ 计算经度合成（矢量化版本 - 高性能）
   事件

In [8]:
# 💾 保存LHF合成数据到fig09目录
print("\n💾 保存LHF合成数据...")

output_dir = "/work/mh1498/m301257/Plot_data/fig09"
os.makedirs(output_dir, exist_ok=True)

# 保存三个实验的LHF合成数据
lhf_save_dict = {
    'CNTL': lhf_composite_cntl,
    'p4k': lhf_composite_p4k,
    '4co2': lhf_composite_4co2
}

diff_qa_qs_save_dict = {
    'CNTL': diff_qs_qa_kelvin_cntl,
    'p4k': diff_qs_qa_kelvin_p4k,
    '4co2': diff_qs_qa_kelvin_4co2
}

surface_wind_save_dict = {
    'CNTL': surface_wind_kelvin_cntl,
    'p4k': surface_wind_kelvin_p4k,
    '4co2': surface_wind_kelvin_4co2
}


for exp_name, lhf_data in lhf_save_dict.items():
    if lhf_data is not None:
        lhf_save_path = os.path.join(output_dir, f'lhf_lon_composite_mean_{exp_name}.nc')
        diff_qa_qs_save_path = os.path.join(output_dir, f'diff_qa_qs_lon_composite_mean_{exp_name}.nc')
        surface_wind_save_path = os.path.join(output_dir, f'surface_wind_lon_composite_mean_{exp_name}.nc')
        diff_qa_qs_data = diff_qa_qs_save_dict[exp_name]
        surface_wind_data = surface_wind_save_dict[exp_name]
        diff_qa_qs_data.to_netcdf(diff_qa_qs_save_path)
        surface_wind_data.to_netcdf(surface_wind_save_path)
        lhf_data.to_netcdf(lhf_save_path)
        file_size = os.path.getsize(lhf_save_path) / 1024
        print(f"  ✅ {exp_name}: 已保存 ({file_size:.1f} KB)")
    else:
        print(f"  ❌ {exp_name}: 数据为空")

print(f"\n📁 数据已保存到: {output_dir}")
print("\n🎉 现在可以使用 fig09_clean.ipynb 进行绘图了！")


💾 保存LHF合成数据...
  ✅ CNTL: 已保存 (9.1 KB)
  ✅ p4k: 已保存 (9.1 KB)
  ✅ 4co2: 已保存 (9.1 KB)

📁 数据已保存到: /work/mh1498/m301257/Plot_data/fig09

🎉 现在可以使用 fig09_clean.ipynb 进行绘图了！
